# Explore a WritingRing recording

This notebook uses the editably installed `writingring` package for discovery, selection, loading, validation, summaries, and plotting. It does not parse filenames, deserialize Board chunks, or reproduce plotting logic locally.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from writingring import (
    build_recording_summary,
    discover_recordings,
    format_recording_summary,
    load_board,
    load_ring,
    plot_board_force_over_time,
    plot_ring_imu,
    plot_touch_trajectory,
    select_recording,
    to_jsonable,
)

## Resolve project data and output paths

The project root lookup makes the notebook work whether execution starts in the repository root or in `notebooks/`. Outputs go under `outputs/notebook_exploration`; source data remains read-only.

In [ ]:
def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "data_sample" / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the project root containing pyproject.toml and data_sample/data")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_ROOT = PROJECT_ROOT / "data_sample" / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "notebook_exploration"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data root: {DATA_ROOT}")
print(f"Notebook outputs: {OUTPUT_DIR}")

## Discover available recordings

In [ ]:
recordings = discover_recordings(DATA_ROOT)
discovery_rows = []
for item in recordings:
    chunk_range = item.chunk_index_range
    discovery_rows.append(
        {
            "user": item.user,
            "action": item.action,
            "dataset_id": item.dataset_id,
            "primary_ring_filename": item.ring_0_path.name,
            "board_chunk_count": len(item.board_chunk_paths),
            "chunk_index_range": None if chunk_range is None else f"{chunk_range[0]}-{chunk_range[1]}",
            "discovery_warnings": "; ".join(item.warnings) if item.warnings else "none",
        }
    )

recordings_table = pd.DataFrame(discovery_rows)
display(recordings_table)

## Select and load the documented example

Dataset 0 is selected through the reusable selector. `load_ring(recording)` loads only the recording's primary `ring_0` payload; `ring_1` remains optional inspection metadata.

In [ ]:
recording = select_recording(recordings, user="user_0", action="0", dataset_id=0)
ring_data = load_ring(recording)
board_data = load_board(recording)

assert ring_data.source_path == recording.ring_0_path
assert board_data.chunk_paths == recording.board_chunk_paths
assert tuple(report.chunk_index for report in board_data.chunk_reports) == tuple(range(16))

print(f"Selected: {recording.user} / action {recording.action} / dataset {recording.dataset_id}")
print(f"Primary Ring payload loaded: {ring_data.source_path.name}")
print(f"Ring 1 metadata only: {recording.ring_1_path.name if recording.ring_1_path else 'absent'}")
print(f"Board chunks retained in numeric order: {len(board_data.chunk_reports)} (indices 0-15)")

## Common recording summary

In [ ]:
summary = build_recording_summary(recording, ring_data, board_data)
print(format_recording_summary(summary))

## Concise data previews

Only the first rows are displayed to avoid embedding large outputs.

In [ ]:
print("Ring rows")
display(ring_data.dataframe.head())

print("Board frame rows")
display(board_data.frames.head())

print("Board contact rows")
display(board_data.contacts.head())

## Structured validation information

The validation objects below come directly from the loaders. Detailed per-channel and per-chunk structures are displayed as compact tables.

In [ ]:
ring_validation = to_jsonable(ring_data.validation)
ring_overview_keys = [
    "sample_count", "value_count", "file_size_bytes", "raw_timestamp_start",
    "raw_timestamp_end", "raw_timestamp_duration", "timestamps_finite",
    "timestamps_nondecreasing", "timestamps_strictly_increasing",
    "duplicate_timestamp_steps", "backward_timestamp_steps",
    "inferred_timestamp_unit", "inferred_duration_s",
    "inferred_sampling_rate_hz", "warnings",
]
display(pd.Series({key: ring_validation[key] for key in ring_overview_keys}, name="Ring validation"))
display(pd.DataFrame(ring_validation["per_channel_statistics"]).T)

In [ ]:
board_validation = to_jsonable(board_data.validation)
board_overview_keys = [
    "chunk_count", "chunk_indices", "missing_chunk_indices", "empty_chunk_indices",
    "frame_count", "contact_count", "frames_without_contacts",
    "first_frame_timestamp_in_numeric_order", "last_frame_timestamp_in_numeric_order",
    "within_chunk_backward_steps", "cross_chunk_backward_boundaries",
    "x_range", "y_raw_range", "force_range", "warnings",
]
display(pd.Series({key: board_validation[key] for key in board_overview_keys}, name="Board validation"))
display(pd.DataFrame(to_jsonable(board_data.chunk_reports)))

## Ring IMU plots

The inferred-time view uses the loader's explicitly labeled, unconfirmed microsecond interpretation. The sample-index view makes no time-unit interpretation. Saved figures are displayed and then closed to release Matplotlib resources.

In [ ]:
ring_inferred_path = OUTPUT_DIR / "ring_imu_inferred_time.png"
figure = plot_ring_imu(ring_data, time_axis="inferred_time", output_path=ring_inferred_path, show=False)
display(figure)
plt.close(figure)

In [ ]:
ring_index_path = OUTPUT_DIR / "ring_imu_sample_index.png"
figure = plot_ring_imu(ring_data, time_axis="sample_index", output_path=ring_index_path, show=False)
display(figure)
plt.close(figure)

## Board plots

The touch trajectory uses the upstream display transformation while preserving `y_raw`. Both Board force views preserve contact row order and expose Dataset 0's timestamp discontinuity rather than repairing it.

In [ ]:
touch_path = OUTPUT_DIR / "touch_trajectory.png"
figure = plot_touch_trajectory(board_data, output_path=touch_path, show=False)
display(figure)
plt.close(figure)

In [ ]:
board_frame_path = OUTPUT_DIR / "board_force_frame_index.png"
figure = plot_board_force_over_time(
    board_data, time_axis="frame_index", output_path=board_frame_path, show=False
)
display(figure)
plt.close(figure)

In [ ]:
board_timestamp_path = OUTPUT_DIR / "board_force_raw_timestamp.png"
figure = plot_board_force_over_time(
    board_data, time_axis="raw_timestamp", output_path=board_timestamp_path, show=False
)
display(figure)
plt.close(figure)

## Independent filtering examples

Board contacts can be filtered by numeric chunk index without changing or rerunning the loader. This example creates a separate view and leaves `board_data.contacts` untouched.

In [ ]:
selected_chunk_indices = (5, 6, 7)
board_contacts_by_chunk = board_data.contacts.loc[
    board_data.contacts["chunk_index"].isin(selected_chunk_indices)
]
print(f"Contacts in numeric Board chunks {selected_chunk_indices}: {len(board_contacts_by_chunk)}")
display(board_contacts_by_chunk.head())

A Ring interval can likewise be selected through the DataFrame's named `sample_index` index. This is an independent Ring-only view.

In [ ]:
ring_sample_start, ring_sample_stop = 1000, 1100
ring_sample_interval = ring_data.dataframe.loc[ring_sample_start:ring_sample_stop]
print(f"Ring sample-index interval {ring_sample_start}-{ring_sample_stop}: {len(ring_sample_interval)} rows")
display(ring_sample_interval.head())

These two filters do **not** establish Ring–Board synchronization. A Board chunk index and a Ring sample index belong to separate streams, and the available data does not document a shared clock or alignment guarantee.

## Execution checks

In [ ]:
saved_figure_paths = [
    ring_inferred_path,
    ring_index_path,
    touch_path,
    board_frame_path,
    board_timestamp_path,
]
boundary_warnings = [warning for warning in board_data.warnings if "6->7" in warning]

assert ring_data.source_path.name.endswith("_ring_0.bin")
assert len(board_data.chunk_reports) == 16
assert tuple(report.chunk_index for report in board_data.chunk_reports) == tuple(range(16))
assert boundary_warnings
assert all(path.is_file() and path.stat().st_size > 0 for path in saved_figure_paths)

display(
    pd.DataFrame(
        {
            "output_file": [path.name for path in saved_figure_paths],
            "size_bytes": [path.stat().st_size for path in saved_figure_paths],
        }
    )
)
print(f"Visible Dataset 0 boundary warning: {boundary_warnings[0]}")
print("Verified: all 16 numeric Board chunks remain represented; no timestamp repair or synchronization was applied.")

## Data limitations and interpretation notes

- Dataset 0 has a backward Board timestamp boundary from chunk `6->7`. Numeric filename order is preserved, including chunks 7–15; timestamps are not repaired and the recording is not split.
- Dataset 0 begins with empty Board chunk 0. Empty chunks remain represented and are disclosed by validation and plot warnings.
- Ring relative time is inferred from an observed microsecond interpretation of the raw timestamp. Microseconds are not a confirmed upstream contract, and the original timestamp remains available.
- Physical units for Ring signals and Board coordinates/force are undocumented, so plots label them as raw or undocumented values.
- Ring–Board synchronization is not guaranteed. The streams are inspected independently; neither the filters nor the plots establish alignment.